# 07 Advanced Challenge - IC50-style Curve Fitting in Python

## Biochemistry question

In this synthetic dose-response dataset, what educational IC50-style pattern is suggested by a simple fitted curve?

This is a learning example, not a production pharmacology workflow.


In [6]:
import plotly.io as pio
pio.renderers.default = "iframe"


In [8]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go

df = pd.read_csv("../data/dose_response_ic50_sample.csv")
df.head()

,sample_id,drug_name,concentration_uM,replicate,cell_viability_percent
0,A001,CompoundX,0.000,1,100
1,A002,CompoundX,0.000,2,98
2,A003,CompoundX,0.000,3,101
3,A004,CompoundX,0.003,1,97
4,A005,CompoundX,0.003,2,95


In [9]:
summary = (
    df.groupby(["drug_name", "concentration_uM"])
      .agg(mean_viability=("cell_viability_percent", "mean"),
           sd_viability=("cell_viability_percent", "std"),
           n=("cell_viability_percent", "count"))
      .reset_index()
)
summary["sem_viability"] = summary["sd_viability"] / np.sqrt(summary["n"])
summary

,drug_name,concentration_uM,mean_viability,sd_viability,n,sem_viability
0,CompoundX,0.000,99.666667,1.527525,3,0.881917
1,CompoundX,0.003,96.000000,1.000000,3,0.577350
2,CompoundX,0.010,90.000000,1.000000,3,0.577350
3,CompoundX,0.030,80.000000,2.000000,3,1.154701
4,CompoundX,0.100,60.666667,1.527525,3,0.881917
5,CompoundX,0.300,41.000000,2.000000,3,1.154701
6,CompoundX,1.000,25.666667,1.527525,3,0.881917
7,CompoundX,3.000,16.000000,1.000000,3,0.577350
8,CompoundY,0.000,100.000000,1.000000,3,0.577350
9,CompoundY,0.003,98.000000,1.000000,3,0.577350


In [4]:
def four_param_logistic(x, bottom, top, ic50, hill):
    return bottom + (top - bottom) / (1 + (x / ic50) ** hill)

fit_results = []

fig = go.Figure()

for drug, group in summary.groupby("drug_name"):
    # Exclude 0 concentration from curve fitting because log/dose curve cannot use x=0 directly.
    fit_group = group[group["concentration_uM"] > 0].copy()
    x = fit_group["concentration_uM"].values
    y = fit_group["mean_viability"].values

    initial_guess = [min(y), max(y), np.median(x), 1.0]
    bounds = ([0, 50, min(x)/10, 0.1], [120, 120, max(x)*10, 5])

    params, _ = curve_fit(
        four_param_logistic,
        x,
        y,
        p0=initial_guess,
        bounds=bounds,
        maxfev=10000
    )

    bottom, top, ic50, hill = params
    fit_results.append({
        "drug_name": drug,
        "bottom": bottom,
        "top": top,
        "estimated_ic50_uM": ic50,
        "hill_slope": hill
    })

    x_pred = np.logspace(np.log10(min(x)), np.log10(max(x)), 200)
    y_pred = four_param_logistic(x_pred, *params)

    fig.add_trace(go.Scatter(
        x=fit_group["concentration_uM"],
        y=fit_group["mean_viability"],
        mode="markers",
        name=f"{drug} observed",
        error_y=dict(type="data", array=fit_group["sem_viability"], visible=True)
    ))

    fig.add_trace(go.Scatter(
        x=x_pred,
        y=y_pred,
        mode="lines",
        name=f"{drug} fitted curve"
    ))

fit_df = pd.DataFrame(fit_results)
fit_df

,drug_name,bottom,top,estimated_ic50_uM,hill_slope
0,CompoundX,8.912777,100.106857,0.142283,0.792885
1,CompoundY,21.185765,100.193041,0.239829,0.875751


In [10]:
fig.update_layout(
    title="IC50-style Curve Fitting Challenge",
    xaxis_title="Concentration (uM, log scale)",
    yaxis_title="Mean Cell Viability (%)",
    template="plotly_white"
)
fig.update_xaxes(type="log")
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## Interpretation Questions

1. Which compound has the lower educational IC50-style estimate?
2. What does a lower estimate suggest within this synthetic example only?
3. Why should this estimate be treated cautiously?
4. What would improve the reliability of the fit?

## Limitations

- This is synthetic data for learning curve-fitting ideas.
- The fitted value is not drug potency, clinical, diagnostic, regulatory, or efficacy evidence.
- Real IC50 analysis requires stronger experimental design, replicate structure, model checking, and domain review.
